In [1]:
from unsloth import FastModel 
# o bicho quer ser importado primeiro

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
COMPUTE_LENGTHS = True
EVAL_BS = 16
MAX_SEQ_LENGTH = 512

# Carregando dados

In [3]:
import pickle
import json
import networkx as nx
import numpy as np

In [4]:
with open('/home/kenzosaki/repos/glm-based-event-analysis/data/processed/directed_mad3.5_ai_news_event_graph.pkl', 'rb') as f:
    G = pickle.load(f)

with open("/home/kenzosaki/repos/glm-based-event-analysis/data/labels/small_directed_mad3.5/link_prediction_edges.json", 'r') as f:
    test_edges = json.load(f)

# Obtendo grafo de treino

In [8]:
def get_G_train(G: nx.Graph, eval_edges: list[tuple]) -> nx.Graph:

    # removendo arestas de teste e nós alvo 
    G_train = G.copy()

    for edge_info in eval_edges:
        # remover apenas arestas que existem
        v = edge_info["v"]

        if edge_info["label"] == 'negative': continue # se a aresta é negativa, nao removemos o nó alvo, pois ele pode aparecer na vizinhança de outros nós

        # remoção do nó (garante que ele nao apareça na vizinhança)
        # remover o no alvo remove a aresta junto
        if G_train.has_node(v):
            G_train.remove_node(v)
    
    return G_train

In [9]:
G_train = get_G_train(G, test_edges)

In [10]:
G_train.number_of_nodes(), G_train.number_of_edges(), nx.number_connected_components(G_train.to_undirected())

(14448, 177138, 2)

# Carregando modelo e tokenizador

In [11]:
from unsloth.chat_templates import get_chat_template

In [12]:
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = MAX_SEQ_LENGTH, # TODO: calibrar esse trem
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

==((====))==  Unsloth 2026.5.1: Fast Gemma3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.671 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [13]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

# Criando exemplos de treino

In [ ]:
from glm_based_event_analysis.link_prediction.datasets import LinkPredictionDatasetBuilder, LinkPredictionDatasetBuilderWithNeighbours
from glm_based_event_analysis.utils.sampling import sample_neg_edges

In [15]:
def sample_edges_for_training(G: nx.Graph, n_edges: int) -> tuple[list[tuple], list[bool]]:
    
    # arestas positivas
    true_edges = list(G.edges(data=False))
    true_edge_sample_idxs = np.random.choice(np.arange(G.number_of_edges()), size=n_edges, replace=False)
    true_edge_samples = [true_edges[i] for i in true_edge_sample_idxs]
    true_edge_sample_labels = [True] * len(true_edge_samples)

    # arestas negativas
    false_edge_samples = sample_neg_edges(G, n_edges)
    false_edge_sample_labels = [False] * len(false_edge_samples)    

    # combinando
    all_edge_samples = true_edge_samples + false_edge_samples
    all_edge_sample_labels = true_edge_sample_labels + false_edge_sample_labels

    return all_edge_samples, all_edge_sample_labels



In [16]:
uv_test_edges = [ (edge_info["u"], edge_info["v"]) for edge_info in test_edges]
test_labels = [edge_info["label"] for edge_info in test_edges]

In [17]:
train_edges_sample, train_labels = sample_edges_for_training(G_train, n_edges=1000)

In [18]:
builder = LinkPredictionDatasetBuilder(tokenizer = tokenizer, G_ref = G)

In [19]:
train_ds = builder.build_train_ds(train_edges_sample, train_labels)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [20]:
test_ds = builder.build_test_ds(uv_test_edges, test_labels) 

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

In [21]:
idx = np.random.choice(len(train_ds))
print(train_ds[idx]['text'])

<start_of_turn>user
Given the following pair of events, determine if there is a link between them based on their components.

Event u:
{
  "what": "Microsoft stock presents an investment opportunity due to a discrepancy between spending and revenue.",
  "where": "United States",
  "when": "2026-03-20-07:00:00",
  "who": "Microsoft, Investors, Seeking Alpha",
  "how": "Investors are advised to analyze the spending versus revenue gap in Microsoft's financial reports to identify potential investment strategies.",
  "why": "A significant gap between Microsoft's spending and revenue creates a potential investment opportunity.",
  "event_id": "8895_revenue_spending_gap_2026-03-20"
}

Event v:
{
  "what": "Concerns arise regarding OpenAI's financial stability and potential acquisition by Microsoft.",
  "where": "United States",
  "when": "2026-03-26-07:00:00",
  "who": "OpenAI, Microsoft, investors",
  "how": "Reports and speculation in financial news outlets suggest a potential acquisition b

In [22]:
idx = np.random.choice(len(test_ds))
print(test_ds[idx]['text'])

<start_of_turn>user
Given the following pair of events, determine if there is a link between them based on their components.

Event u:
{
  "what": "Matrix expands its global AI collaboration with Dataiku to the Americas.",
  "where": "United States",
  "when": "2026-03-30-07:00:00",
  "who": "Matrix and Dataiku",
  "how": "Through an expansion of existing global collaboration efforts.",
  "why": "To broaden AI collaboration in the Americas.",
  "event_id": "2847_dataiku_ai_collaboration_2026-03-30"
}

Event v:
{
  "what": "Lovable handles 1 billion tokens per minute.",
  "where": "India",
  "when": "2026-03-09-07:00:00",
  "who": "Lovable",
  "how": "Through advanced analytics and infrastructure.",
  "why": "To process a large volume of data.",
  "event_id": "4360_tokens_lovable_analytics_2026-03-09"
}<end_of_turn>
<start_of_turn>model



# Preparando callback para avaliação entre steps

In [ ]:
from glm_based_event_analysis.link_prediction.callbacks import LinkPredictionEvalCallback

In [25]:
# tokenizando e preparando exemplos
def tokenizer_function(examples):

    input_ids = tokenizer(
        examples["text"],
        max_length=MAX_SEQ_LENGTH, 
        add_special_tokens=True,
        truncation=True
    )

    return input_ids


In [26]:
# preparando o ds de teste
# se der bom, migrar para dentro do callback
test_ds = test_ds.map(tokenizer_function, batched=True, remove_columns=test_ds.column_names)
test_ds = test_ds.with_format("torch")

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

In [ ]:
lp_callback = LinkPredictionEvalCallback(
    eval_ds=test_ds, 
    labels=[True if label == 'positive' else False for label in test_labels],
    tokenizer=tokenizer.tokenizer,
    eval_bs=EVAL_BS,
    log_file="lp_metrics.json"
)

# Fine-tunning

Demo do finetunning

In [27]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import math

In [28]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 2026,
)

In [ ]:
final_ds = train_ds.train_test_split(test_size=0.1, seed=2026, stratify_by_column="label") # TODO: tem um stratify by col. aproveitar dpois

In [30]:
final_bs = 8 * 64
evals_per_epoch = 20
epoch_steps = math.ceil(len(final_ds['train'])/final_bs)
eval_steps = int(epoch_steps / evals_per_epoch)

In [31]:
epoch_steps, eval_steps

(4, 0)

In [32]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 32,
        gradient_accumulation_steps = 16, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        eval_strategy="steps",
        eval_steps = 2,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 2046,
        report_to = "none", # Use TrackIO/WandB etc
        metric_for_best_model="eval_loss",
        # load_best_model_at_end=True,
)


trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = final_ds['train'], # TODO: amostrar um conj de eval daqui
    eval_dataset = final_ds['test'], # Can set up evaluation!
    args = args
)

In [33]:
test_labels[:3]

['positive', 'positive', 'positive']

In [35]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=36):   0%|          | 0/1800 [00:00<?, ? examples/s]

Map (num_proc=36):   0%|          | 0/1800 [00:00<?, ? examples/s]

Filter (num_proc=36):   0%|          | 0/1800 [00:00<?, ? examples/s]

Unsloth: Removed 9 out of 1800 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


Map (num_proc=36):   0%|          | 0/200 [00:00<?, ? examples/s]

Map (num_proc=36):   0%|          | 0/200 [00:00<?, ? examples/s]

Filter (num_proc=36):   0%|          | 0/200 [00:00<?, ? examples/s]

Unsloth: Removed 2 out of 200 samples from eval_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


In [36]:
print(tokenizer.decode(trainer.train_dataset[100]["input_ids"]))

<bos><start_of_turn>user
Given the following pair of events, determine if there is a link between them based on their components.

Event u:
{
  "what": "Google Messages introduces two previously absent basic features.",
  "where": "United States",
  "when": "2026-03-20-07:00:00",
  "who": "Google, Google Messages users",
  "how": "Through a software update to the Google Messages application.",
  "why": "To enhance user experience and provide functionalities previously expected.",
  "event_id": "7270_messages_features_google_2026-03-20"
}

Event v:
{
  "what": "Google is testing a new search feature that replaces website headlines and titles with AI-generated summaries.",
  "where": "United States",
  "when": "2026-03-21-07:00:00",
  "who": "Google",
  "how": "Google is implementing an AI algorithm to generate summaries that replace traditional website titles in search results; this is being tested with a limited user group.",
  "why": "To improve user experience and provide more concis

In [ ]:
# trainer.add_callback(es_callback)
trainer.add_callback(lp_callback)

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,791 | Num Epochs = 1 | Total steps = 4
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 16 x 1) = 512
 "-____-"     Trainable parameters = 14,901,248 of 4,314,980,720 (0.35% trained)
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Unsloth: Will smartly offload gradients to save VRAM!


/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
2,8.202300,8.192912
4,7.337700,5.037334


Evaluating...


- Remaining batches:   0%|          | 0/45 [00:00<?, ?it/s]/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning:

              precision    recall  f1-score   support

       False       0.50      1.00      0.67       356
        True       0.00      0.00      0.00       356

    accuracy                           0.50       712
   macro avg       0.25      0.50      0.33       712
weighted avg       0.25      0.50      0.33       712

Evaluating...


- Remaining batches:   2%|▏         | 1/45 [00:09<07:03,  9.62s/it]

In [ ]:
test_labels

['positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',
 'positive',

# Preparando inferência

In [ ]:
from transformers import AutoTokenizer, DataCollatorForSeq2Seq, DataCollatorForSeq2Seq
from torch.utils.data import DataLoader

In [ ]:
def get_collator(tokenizer: AutoTokenizer) -> DataCollatorForSeq2Seq:

    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding="longest",
        label_pad_token_id=tokenizer.pad_token_id
    )

    return collator

In [ ]:
test_ds = test_ds.map(tokenizer_function, batched=True, remove_columns=test_ds.column_names)

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

In [ ]:
test_ds = test_ds.with_format("torch")

In [ ]:
collator = get_collator(tokenizer.tokenizer) # o tokenizador do gemma tem dois componentes/modalidades
inference_dl = DataLoader(test_ds, batch_size=EVAL_BS, collate_fn=collator)

# Inferencia

In [ ]:
import torch
from tqdm import tqdm

In [ ]:
trainer.args.device

device(type='cuda', index=0)

In [ ]:
model = FastModel.for_inference(model)

In [ ]:
preds = []
with torch.inference_mode():
    for batch in tqdm(inference_dl, desc="- Remaining batches", leave=False):

        batch = batch.to(trainer.args.device)
        
        generated_ids = model.generate(
            **batch,
            do_sample=False,
            max_new_tokens=64
        )
        
        decoded_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        preds.extend(decoded_texts)

        del batch


- Remaining batches:   0%|          | 0/45 [00:00<?, ?it/s]/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/kenzosaki/micromamba/envs/event_analysis/lib/python3.12/site-packages/bitsandbytes/_ops.py:186: FutureWarning:

In [ ]:
preds[:3]


['user\nGiven the following pair of events, determine if there is a link between them based on their components.\n\nEvent u:\n{\n  "what": "Nebius secures a $4.3 billion debt raise to fund its AI initiatives.",\n  "where": "United States",\n  "when": "2026-03-23-07:00:00",\n  "who": "Nebius",\n  "how": "Through a $4.3 billion debt raise.",\n  "why": "To gain a competitive advantage in the AI race.",\n  "event_id": "9107_ai_nebius_debt_2026-03-23"\n}\n\nEvent v:\n{\n  "what": "Mistral secures $830 million in debt financing for an AI data center in Paris utilizing Nvidia chips.",\n  "where": "France",\n  "when": "2026-03-30-07:00:00",\n  "who": "Mistral, Nvidia",\n  "how": "Through the acquisition of $830 million in debt financing.",\n  "why": "To expand AI infrastructure and processing capabilities.",\n  "event_id": "10850_mistral_debt_paris_2026-03-30"\n}\nmodel\n3 true',
 'user\nGiven the following pair of events, determine if there is a link between them based on their components.\n\

In [ ]:
raw_preds = []
for pred in preds:
    pred_json_str = pred.split("model\n")[-1].strip() # extrai a parte gerada pelo modelo
    raw_preds.append(pred_json_str)

final_preds = []
for raw_pred in raw_preds:
    if "true" in raw_pred.lower():
        final_preds.append(True)
    elif "false" in raw_pred.lower():
        final_preds.append(False)
    else:
        final_preds.append(False) # ou algum valor padrão para casos ambíguos

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
test_labels = [True if label == 'positive' else False for label in test_labels]

In [ ]:
print(classification_report(test_labels, final_preds))

              precision    recall  f1-score   support

       False       0.55      0.97      0.70       356
        True       0.87      0.20      0.33       356

    accuracy                           0.59       712
   macro avg       0.71      0.59      0.51       712
weighted avg       0.71      0.59      0.51       712

